In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# load data
reviews = pd.read_csv('/Users/eggy/Downloads/reviews.csv.gz', compression='gzip')
listings_clean = pd.read_csv('data/listings_clean.csv')

print(f'Reviews: {reviews.shape}')
print(f'Listings: {listings_clean.shape}')

Reviews: (977459, 6)
Listings: (36613, 23)


In [42]:
# basic stats
print('Reviews per listing:')
print(reviews.groupby('listing_id')['id'].count().describe())

print('\nDate range:')
print(f'Earliest review: {reviews["date"].min()}')
print(f'Latest review: {reviews["date"].max()}')

print('\nMissing comments:')
print(reviews['comments'].isna().sum())

print('\nSample reviews:')
print(reviews['comments'].dropna().sample(5, random_state=42).tolist())

Reviews per listing:
count    25093.000000
mean        38.953453
std         79.458967
min          1.000000
25%          3.000000
50%         11.000000
75%         42.000000
max       3518.000000
Name: id, dtype: float64

Date range:
Earliest review: 2009-05-25
Latest review: 2025-08-02

Missing comments:
260

Sample reviews:
['Fantastic to get to live in Williamsburg at A and A’s place! We loved the surroundings and our hosts. They were sweet, nice, helpful, inviting people! Even the neighbours were nice. We had good talks on the wonderful patio.<br/>The room and private bathrooms was fine and clean and with a good WiFi ;-) The bed was a little soft... but equipped with tons of good pillows :-) <br/>It’s SO easy (short walk) to get to the subway - and only a short train ride to Manhattan. We enjoyed Graham Avenue’s shops, restaurants, a great barber shop - and the quiet neighbourhood. ', 'We loved staying here!  It was beyond what we expected, clean, comfortable and so well done!  Lo

In [28]:
# drop missing comments
reviews = reviews.dropna(subset=['comments'])

# clean HTML tags
reviews['comments'] = reviews['comments'].str.replace('<br/>', ' ', regex=False)
reviews['comments'] = reviews['comments'].str.replace('<[^>]+>', ' ', regex=True)

# merge with listings
merged = reviews.merge(
    listings_clean[['id', 'instant_bookable', 'neighbourhood_group_cleansed',
                    'room_type', 'estimated_occupancy_l365d', 'price']],
    left_on='listing_id',
    right_on='id',
    how='inner'
)

print(f'Reviews after merge: {merged.shape}')
print(f'Unique listings in merged: {merged["listing_id"].nunique()}')

Reviews after merge: (936325, 12)
Unique listings in merged: 23976


In [30]:
# review stats
merged['review_stats'] = merged['comments'].str.len()

print('Review stats:')
print(merged['review_stats'].describe())

# reviews by year
merged['date'] = pd.to_datetime(merged['date'])
merged['year'] = merged['date'].dt.year

print('\nReviews by year:')
print(merged['year'].value_counts().sort_index())

Review stats:
count    936325.000000
mean        253.214663
std         251.741871
min           1.000000
25%          86.000000
50%         183.000000
75%         336.000000
max        5921.000000
Name: review_stats, dtype: float64

Reviews by year:
year
2009        43
2010       416
2011      1632
2012      3229
2013      5865
2014     11264
2015     23056
2016     39896
2017     54965
2018     78602
2019    101845
2020     39131
2021     80502
2022    137689
2023    157147
2024    124488
2025     76555
Name: count, dtype: int64


In [44]:
print('\nColumns Before Merge:')
print(reviews.columns.tolist())

print('\nMerged Columns:')
print(merged.columns.tolist())
print(merged[['listing_id', 'id_x', 'id_y']].head())

# null check
print('\nNull values in merged dataset:')
print(merged.isnull().sum())


Columns Before Merge:
['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']

Merged Columns:
['listing_id', 'id_x', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'id_y', 'instant_bookable', 'neighbourhood_group_cleansed', 'room_type', 'estimated_occupancy_l365d', 'price', 'review_stats', 'year']
   listing_id       id_x  id_y
0        2539   55688172  2539
1        2539   97474898  2539
2        2539  105340344  2539
3        2539  133131670  2539
4        2539  138349776  2539

Null values in merged dataset:
listing_id                      0
id_x                            0
date                            0
reviewer_id                     0
reviewer_name                   4
comments                        0
id_y                            0
instant_bookable                0
neighbourhood_group_cleansed    0
room_type                       0
estimated_occupancy_l365d       0
price                           0
review_stats                    0
year             

In [40]:
# reviews per listing -- are reviews concentrated on a few listings?
reviews_per_listing = merged.groupby('listing_id')['id_x'].count()
print(reviews_per_listing.describe())
print(f'\nTop 5 most reviewed listings:')
print(reviews_per_listing.sort_values(ascending=False).head())


count    23976.000000
mean        39.052594
std         80.075995
min          1.000000
25%          3.000000
50%         11.000000
75%         42.000000
max       3517.000000
Name: id_x, dtype: float64

Top 5 most reviewed listings:
listing_id
858697692672545141    3517
593322347340602809    2831
51619634              2141
37122502              1960
691676460109271194    1903
Name: id_x, dtype: int64


Review distribution is highly skewed -- median 11 reviews per listing 
but max 3,517. Listing-level sentiment will be aggregated as a mean 
across all reviews, so heavily reviewed listings will have more stable 
sentiment estimates than listings with only 1-2 reviews.

In [16]:
# save reviews for Colab sentiment scoring
merged.to_csv('data/reviews_merged.csv', index=False)
print(f'Saved {len(merged)} reviews to reviews_merged.csv')

Saved 936325 reviews to reviews_merged.csv


In [45]:
sentiment = pd.read_csv('data/sentiment_scores.csv')
print(f'Loaded {len(sentiment)} sentiment scores')
print(sentiment.head())
print(sentiment['sentiment_label'].value_counts())

Loaded 400000 sentiment scores
   listing_id sentiment_label  sentiment_score
0    16396241        POSITIVE         0.999681
1     4268286        POSITIVE         0.998254
2    27598707        POSITIVE         0.999803
3    49274169        POSITIVE         0.999661
4    54247180        POSITIVE         0.999417
sentiment_label
POSITIVE    350163
NEGATIVE     49837
Name: count, dtype: int64


In [46]:
# aggregate sentiment to listing level
listing_sentiment = sentiment.groupby('listing_id').agg(
    avg_sentiment_score=('sentiment_score', 'mean'),
    pct_positive=('sentiment_label', lambda x: (x == 'POSITIVE').mean()),
    review_count=('sentiment_label', 'count')
).reset_index()

print(f'Listings with sentiment: {len(listing_sentiment)}')
print(listing_sentiment.describe())

Listings with sentiment: 20847
         listing_id  avg_sentiment_score  pct_positive  review_count
count  2.084700e+04         20847.000000  20847.000000  20847.000000
mean   3.075232e+17             0.984933      0.881889     19.187413
std    4.444000e+17             0.032940      0.193021     36.218731
min    2.539000e+03             0.512411      0.000000      1.000000
25%    1.743458e+07             0.982095      0.833333      2.000000
50%    4.174664e+07             0.996463      0.959184      7.000000
75%    7.252820e+17             0.999690      1.000000     22.000000
max    1.409437e+18             0.999894      1.000000   1545.000000


In [47]:
# merge with listings
listings_sentiment = listings_clean.merge(
    listing_sentiment,
    left_on='id',
    right_on='listing_id',
    how='left'
)

print(f'Listings with sentiment: {listings_sentiment["avg_sentiment_score"].notna().sum()}')
print(f'Listings without sentiment: {listings_sentiment["avg_sentiment_score"].isna().sum()}')

# correlation with occupancy
corr = listings_sentiment[['avg_sentiment_score', 'pct_positive', 
                            'estimated_occupancy_l365d']].corr()
print('\nCorrelation with occupancy:')
print(corr['estimated_occupancy_l365d'])

Listings with sentiment: 20847
Listings without sentiment: 15766

Correlation with occupancy:
avg_sentiment_score         -0.037233
pct_positive                -0.051680
estimated_occupancy_l365d    1.000000
Name: estimated_occupancy_l365d, dtype: float64


Sentiment shows weak negative correlation with occupancy (-0.037 to -0.052).
High occupancy listings do not necessarily have better sentiment -- 
possibly because popular listings attract more reviews including 
more critical ones, or because structured features like review scores 
already capture listing quality more precisely than text sentiment.

In [48]:
ib_sentiment = listings_sentiment.groupby('instant_bookable').agg(
    avg_sentiment=('avg_sentiment_score', 'mean'),
    pct_positive=('pct_positive', 'mean'),
    count=('id', 'count')
).reset_index()

print(ib_sentiment)

   instant_bookable  avg_sentiment  pct_positive  count
0             False       0.985717      0.891191  29108
1              True       0.981509      0.841265   7505


In [49]:
room_sentiment = listings_sentiment.groupby('room_type').agg(
    avg_sentiment=('avg_sentiment_score', 'mean'),
    pct_positive=('pct_positive', 'mean'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    count=('id', 'count')
).reset_index()

print(room_sentiment)

         room_type  avg_sentiment  pct_positive  avg_occupancy  count
0  Entire home/apt       0.985928      0.889760      49.368466  19706
1       Hotel room       0.980416      0.863521      10.496945    491
2     Private room       0.983707      0.871866      44.635401  16237
3      Shared room       0.981781      0.875666      34.072626    179


In [50]:
borough_sentiment = listings_sentiment.groupby('neighbourhood_group_cleansed').agg(
    avg_sentiment=('avg_sentiment_score', 'mean'),
    pct_positive=('pct_positive', 'mean'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    count=('id', 'count')
).reset_index()

print(borough_sentiment)

  neighbourhood_group_cleansed  avg_sentiment  pct_positive  avg_occupancy  \
0                        Bronx       0.984198      0.881210      51.835720   
1                     Brooklyn       0.987211      0.905621      48.081224   
2                    Manhattan       0.982886      0.858490      43.055454   
3                       Queens       0.984652      0.883004      51.914329   
4                Staten Island       0.986529      0.895412      64.591160   

   count  
0   1187  
1  13432  
2  16356  
3   5276  
4    362  


Room type sentiment patterns align with Project 1 occupancy findings --
entire homes show both highest sentiment and highest occupancy, while 
hotel rooms show lowest on both metrics. At borough level, Brooklyn and 
Queens show strong sentiment consistent with their significant occupancy 
lifts found in Project 1 subgroup analysis. Manhattan's lower sentiment 
despite high demand suggests guests have higher expectations in premium 
markets.